### Extraction of Data for plotting of MP2-cylinder-paper data.
- create index table which contains all simulations with its parameters, stats and the status if drag and lift timeseries are available
- extract chosen data timeseries for drag and lift coefficient and save them in a new folder with all relevant parameters in their name
- FUTURE: do statistics on data, pre-analyze (plot), incorporate data in plots of cylinder-paper

#### IMPORTS

In [1]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path
import time

#### SCRIPT: Extract INDEX TABLE

rsync -avzP --dry-run \
  --exclude='*h3d*' \
  --exclude='*v3d*' \
  --exclude='*sparse_variants*' \
  --exclude='*VRAMlimitTest*' \
  --exclude='*noForce*' \
  --exclude='*NaN_interval*' \
  --exclude='animations/' \
  --exclude='vtk/' \
  --exclude='*.png' \
  --exclude='observable_2D_plots/' \
  --exclude='watchdog/' \
  --exclude='PU_point_report/' \
  --exclude='HighMaReporter/' \
  --exclude='*GPU_counted_tensors*' \
  --exclude='*GPU_list_of_tensors*' \
  --exclude='*GPU_memory*' \
  --exclude='*t-avg*' \
  --exclude='*test_A100*' \
  --exclude='*obstacle_mask_info.txt*' \
  --exclude='AvgVelocity_Data/' \
  --exclude='*_sparse_*' \
  --exclude='*.cpt' \
  mbille3s@wr0:/work/mbille3s/21_LBM/01_data/ . 2>&1 | tee rsync_testlauf.txt

#### FILE LISTE PARSEN UND INDEX TABELLE

In [33]:
def parse_stats_file(file_path):
    content = file_path.read_text()
    data = {}

    # Helfer für verschiedene Datentypen
    def search(pattern, text, dtype=float):
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            val = match.group(1).strip()
            return dtype(val) if dtype else val
        return None

    # --- Parameter ---
    data['Re'] = search(r'Re\s*=\s*([\d\.]+)', content)
    data['Ma'] = search(r'Ma\s*=\s*([\d\.]+)', content)
    data['n_steps'] = search(r'n_steps\s*=\s*(\d+)', content, int)
    data['T_target'] = search(r'T_target\s*=\s*([\d\.]+)', content)
    data['GPD'] = search(r'gridpoints_per_diameter \(gpd\)\s*=\s*(\d+)', content, int)

    # Geometrie & Grid
    data['DpX'] = search(r'DpX \(D/X\)\s*=\s*([\d\.]+)', content)
    data['DpY'] = search(r'DpY \(D/Y\)\s*=\s*([\d\.]+)', content)
    data['DpZ'] = search(r'DpZ \(D/Z\)\s*=\s*([\d\.]+)', content)
    data['shape_LU'] = search(r'shape_LU:\s*\((.*?)\)', content, str)
    data['gridpoints'] = search(r'total_number_of_gridpoints:\s*(\d+)', content, int)

    # Numerik
    data['bc_type'] = search(r'bc_type\s*=\s*(\w+)', content, str)
    data['stencil'] = search(r'stencil\s*=\s*(\w+)', content, str)
    data['collision'] = search(r'collision\s*=\s*(\w+)', content, str)
    data['tau'] = search(r'tau\s*=\s*([\d\.]+)', content)

    # --- Performance ---
    data['runtime'] = search(r'runtime\s*=\s*([\d\.]+)', content)
    data['MLUPS'] = search(r'MLUPS\s*=\s*([\d\.]+)', content)
    data['VRAM_peak'] = search(r'VRAM_peak \[MB\]\s*=\s*([\d\.]+)', content)

    return data

def build_index(base_dir):
    base_path = Path(base_dir)
    records = []

    # Wir suchen nach der primären Stats-Datei
    for stats_file in base_path.rglob("*_parms_stats_obs.txt"):
        folder = stats_file.parent

        # Basis-Info
        entry = {
            'Folder': folder.name,
            'Timestamp': stats_file.name[:13], # yymmdd_hhmmss
        }

        # Werte aus Datei extrahieren
        file_data = parse_stats_file(stats_file)
        entry.update(file_data)

        # Check auf Vorhandensein der Zeitreihen
        # Sucht flexibel nach Dateien die "drag" oder "lift" im Namen haben
        entry['has_drag'] = any(folder.glob("drag*.txt"))
        entry['has_lift'] = any(folder.glob("lift*.txt"))

        records.append(entry)

    df = pd.DataFrame(records)

    # Spalten sortieren für bessere Lesbarkeit
    cols = ['Timestamp', 'Re', 'Ma', 'n_steps', 'T_target', 'GPD',
            'DpX', 'DpY', 'DpZ', 'shape_LU', 'gridpoints',
            'bc_type', 'stencil', 'collision', 'tau',
            'runtime', 'MLUPS', 'VRAM_peak', 'has_drag', 'has_lift', 'Folder']

    # Nur vorhandene Spalten nehmen (falls mal ein Wert in allen Dateien fehlt)
    existing_cols = [c for c in cols if c in df.columns]
    return df[existing_cols]

# Nutzung:
# index_df = build_index("./mein_lokaler_ordner")
# index_df.to_csv("sim_index.csv", index=False)

#### INDEX checken und eine duplikat-Spalte anhängen, falls Simulationen in Kernparametern identisch sind.

In [34]:
# 260121_150000
def build_index_with_check(base_dir):
    df = build_index(base_dir) # Unser bisheriges Skript

    # Markiere Zeilen, die in den Kernparametern identisch sind
    core_params = ['Re', 'Ma', 'GPD', 'stencil', 'collision', 'shape_LU', 'bc_type']
    df['is_duplicate'] = df.duplicated(subset=core_params, keep=False)

    return df

In [35]:
# 260121_151000
def get_time_range(file_path):
    """Liest den ersten und letzten timePU-Wert aus einer Datei mit Header."""
    try:
        with open(file_path, 'rb') as f:
            # 1. Header überspringen
            f.readline()

            # 2. Erste Datenzeile lesen
            first_data_line = f.readline().decode().split()
            if not first_data_line:
                return None, None

            # Laut deiner Info: Spalte 0=stepLU, 1=timePU, 2=Cl/Cd
            time_start = float(first_data_line[1])

            # 3. Zur letzten Zeile springen (effizient bei großen Dateien)
            f.seek(0, os.SEEK_END)
            file_size = f.tell()
            pointer = file_size - 2

            # Rückwärts suchen bis zum vorletzten Zeilenumbruch
            while pointer > 0:
                f.seek(pointer)
                if f.read(1) == b'\n':
                    break
                pointer -= 1

            last_line = f.readline().decode().split()
            # Falls die letzte Zeile leer ist, ggf. die davor nehmen
            if not last_line or len(last_line) < 2:
                # Einfachheitshalber hier ein Fallback für sehr kleine Files
                return time_start, time_start

            time_end = float(last_line[1])
            return time_start, time_end
    except Exception as e:
        # print(f"Info: Konnte Zeitbereich in {file_path.name} nicht lesen: {e}")
        return None, None

def build_complete_index(base_dir):
    base_path = Path(base_dir)
    records = []

    # Suche alle stats-Dateien
    stats_files = list(base_path.rglob("*_parms_stats_obs.txt"))
    print(f"Analysiere {len(stats_files)} Simulationsordner...")

    for stats_file in stats_files:
        folder = stats_file.parent

        # Basis-Parameter extrahieren (Logik wie zuvor)
        entry = {
            'Timestamp': stats_file.name[:13],
            'Folder': folder.name,
            # ... hier alle anderen Parameter aus dem vorherigen Skript einfügen ...
        }

        # Zeitbereich aus drag-Datei (falls vorhanden)
        drag_files = list(folder.glob("drag*.txt"))
        if drag_files:
            t_start, t_end = get_time_range(drag_files[0])
            entry['time_start'] = t_start
            entry['time_end'] = t_end
            entry['has_drag'] = True
            entry['has_lift'] = any(folder.glob("lift*.txt"))
        else:
            entry['time_start'] = entry['time_end'] = None
            entry['has_drag'] = entry['has_lift'] = False

        records.append(entry)

    df = pd.DataFrame(records)

    # Hilfreiche Sortierung: Erst nach Parametern, dann nach Zeit
    # (Ersetze 'Re', 'Ma', 'GPD' durch deine Spaltennamen)
    # sort_cols = [c for c in ['Re', 'Ma', 'GPD', 'time_start'] if c in df.columns]
    # df = df.sort_values(by=sort_cols)

    return df

In [36]:
# ZEIT-RANGES, Überlapps und Lücken identifizieren
def analyze_simulation_continuity(df_index):
    # Wir gruppieren nach allen Parametern, die eine Simulationsreihe definieren
    # 'Folder' und 'Timestamp' lassen wir weg, da diese variieren
    core_params = ['Re', 'Ma', 'GPD', 'DpX', 'DpY', 'DpZ', 'bc_type', 'stencil', 'collision']

    # Sicherstellen, dass die Spalten im DF existieren
    available_params = [c for c in core_params if c in df_index.columns]

    # Sortieren: Erst nach Parametern, dann nach dem Startzeitpunkt der Zeitreihe
    df_sorted = df_index.sort_values(by=available_params + ['time_start']).copy()

    # Ergebnisse für die Analyse
    analysis_results = []

    for name, group in df_sorted.groupby(available_params):
        group = group.reset_index()

        for i in range(len(group)):
            row = group.iloc[i]
            issues = []

            # 1. Check: Fängt die erste Simulation der Gruppe bei 0 an?
            if i == 0 and row['time_start'] > 0.5: # 0.5 als Toleranz für Initialisierung
                issues.append(f"Start-Lücke: Beginnt erst bei {row['time_start']}")

            # 2. Check: Anschluss an die vorherige Simulation
            if i > 0:
                prev_row = group.iloc[i-1]
                gap = row['time_start'] - prev_row['time_end']

                if gap > 0.1: # Toleranz für Lücken
                    issues.append(f"Lücke zum Vorgänger: {gap:.2f} s")
                elif gap < -0.1: # Überlappung
                    issues.append(f"Überlappung: {abs(gap):.2f} s")

            # 3. Check: Hat die Simulation die Zielzeit erreicht?
            # Wir prüfen das nur für das letzte Element der Kette
            if i == len(group) - 1:
                if row['T_target'] and row['time_end'] < (row['T_target'] - 1.0):
                    issues.append(f"Unvollständig: Endet bei {row['time_end']} statt {row['T_target']}")

            # Ergebnis speichern
            res = row.to_dict()
            res['continuity_issues'] = "; ".join(issues) if issues else "OK"
            res['is_complete_chain'] = len(issues) == 0
            analysis_results.append(res)

    return pd.DataFrame(analysis_results)

# Anwendung nach dem Erstellen des Index:
# df_analysis = analyze_simulation_continuity(index_df)
# df_analysis.to_csv("sim_quality_check.csv", index=False)

In [37]:
# INDEX TABELLE
df_index = build_index("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/00_CP-data_raw")
df_index.to_csv("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/sim_index.csv", index=False)

In [38]:
# INDEX TABELLE um DUplikat-Spalte ergänzt (siehe oben was verglichen wird...)
df_index_withduplicate = build_index_with_check("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/00_CP-data_raw")
df_index_withduplicate.to_csv("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/sim_index_dupl.csv", index=False)

In [39]:
# WEITERE TABELLE MIT ZEIT-RANGES von bis
df_time_intervals = build_complete_index("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/00_CP-data_raw")
df_time_intervals.to_csv("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/sim_index_intervals.csv", index=False)

Analysiere 900 Simulationsordner...


In [2]:
# KONTINUITÄT checken
df_analysis = analyze_simulation_continuity(df_time_intervals)
df_analysis.to_csv("sim_quality_check.csv", index=False)

In [8]:
# 260121_152500 - neues Kontinuitäts-Analyse Skript
def analyze_simulation_continuity(df):
    # Definieren, was eine "identische" Simulation ausmacht
    core_params = ['Re', 'Ma', 'GPD', 'DpX', 'DpY', 'stencil', 'bc_type', 'collision']

    # Sortieren nach Parametern und dann chronologisch nach dem Start der Zeitreihe
    df_sorted = df.sort_values(by=core_params + ['time_start']).copy()

    analysis = []
    # Gruppieren nach Parameter-Sets
    for name, group in df_sorted.groupby(core_params):
        group = group.reset_index(drop=True)

        for i in range(len(group)):
            row = group.iloc[i].to_dict()
            issues = []

            # Check: Fängt die Kette bei 0 an?
            if i == 0 and row['time_start'] > 0.5:
                issues.append(f"Anfang fehlt (startet bei {row['time_start']})")

            # Check: Anschluss an Vorgänger
            if i > 0:
                gap = row['time_start'] - group.iloc[i-1]['time_end']
                if gap > 0.1: issues.append(f"Lücke: {gap:.2f}s")
                if gap < -0.1: issues.append(f"Überlappung: {abs(gap):.2f}s")

            # Check: Zielzeit erreicht? (Nur beim letzten Element der Gruppe)
            if i == len(group) - 1:
                if row['T_target'] and row['time_end'] < (row['T_target'] - 1.0):
                    issues.append(f"Abbruch bei {row['time_end']} (Soll: {row['T_target']})")

            row['Status'] = " / ".join(issues) if issues else "OK"
            analysis.append(row)

    return pd.DataFrame(analysis)

# Anwendung:
# df_master = build_master_index("./lokale_daten")
# df_checked = analyze_simulation_continuity(df_master)
# df_checked.to_csv("qualitaets_check.csv")

In [43]:
# 260121_152700
df_master = build_master_index("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/00_CP-data_raw")
df_checked = analyze_simulation_continuity(df_master)
df_checked.to_csv("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/qualitaets_check.csv")

Verarbeite 900 Simulationen...


In [9]:
# 260121_155100
# 1. Variablen löschen (nur im Notebook nötig)
# %reset -f  # (Optionaler Befehl für Jupyter, um alles zu leeren)

# 2. Den Index komplett neu aufbauen
print("Starte frischen Datei-Scan...")
df_fresh = build_master_index("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/00_CP-data_raw")
timestamp = time.strftime("%H%M%S")
filename = f"df_index_{timestamp}.csv"
df_fresh.to_csv("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/" + filename, index=False)

# 3. Den Qualitäts-Check auf den neuen Daten ausführen
df_checked = analyze_simulation_continuity(df_fresh)

# 4. Speichern unter neuem Namen (um Cache-Probleme auszuschließen)
timestamp = time.strftime("%H%M%S")
filename = f"qualitaets_check_{timestamp}.csv"
df_checked.to_csv("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/" + filename, index=False)

print(f"Analyse abgeschlossen. Datei gespeichert als: {filename}")
print(f"Anzahl gefundener Ordner: {len(df_checked)}")

Starte frischen Datei-Scan...
Verarbeite 893 Simulationen...
Analyse abgeschlossen. Datei gespeichert als: qualitaets_check_160748.csv
Anzahl gefundener Ordner: 893


In [21]:
# 260121_1631 - Master-Daten erstellen, Simulationen aneinandergehängt!
def merge_simulation_chains(df_checked, output_base_dir="./merged_data"):
    output_path = Path(output_base_dir)
    output_path.mkdir(exist_ok=True, parents=True)

    core_params = ['Re', 'Ma', 'GPD', 'DpY', 'stencil', 'bc_type', 'collision']
    df_valid = df_checked[df_checked['has_drag'] == True].copy()

    for name, group in df_valid.groupby(core_params, dropna=False):
        group = group.sort_values('time_start')
        first_row, last_row = group.iloc[0], group.iloc[-1]

        t_end_round = int(round(last_row['time_end']))
        name_parts = [
            f"Re{first_row['Re']}", f"Ma{first_row['Ma']}",
            f"GPD{int(first_row['GPD'] or 0)}", f"{first_row['stencil']}",
            f"DpY{first_row['DpY']}", f"BC_{first_row['bc_type']}",
            f"Coll_{first_row['collision']}"
        ]
        if "D3" in str(first_row['stencil']):
            name_parts.append(f"DpZ{first_row['DpZ']}")

        name_parts.append(f"T{t_end_round}")
        base_filename = "_".join(filter(None, [str(p) for p in name_parts]))

        for mode in ['drag', 'lift']:
            combined_chunks = []
            for _, row in group.iterrows():
                folder = Path(row['path'])
                files = list(folder.glob(f"{mode}*.txt"))
                if files:
                    combined_chunks.append(pd.read_csv(files[0], sep=r'\s+'))

            if not combined_chunks: continue

            full_df = pd.concat(combined_chunks, ignore_index=True)

            # Überlapp-Check (5 Nachkommastellen)
            full_df['time_rounded'] = full_df['timePU'].round(5)
            full_df = full_df.drop_duplicates(subset=['time_rounded'], keep='first')
            full_df = full_df.drop(columns=['time_rounded']).sort_values('timePU')

            # Speichern als CSV statt Parquet
            target_file = output_path / f"{base_filename}_{mode}.csv"
            full_df.to_csv(target_file, index=False)
            print(f"Erfolgreich als CSV gespeichert: {target_file.name}")

In [22]:
merge_simulation_chains(df_checked, "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/01_CP-data_processed")

Erfolgreich als CSV gespeichert: Re10.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_drag.csv
Erfolgreich als CSV gespeichert: Re10.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_lift.csv
Erfolgreich als CSV gespeichert: Re10.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_ibb1c2_Coll_bgk_T300_drag.csv
Erfolgreich als CSV gespeichert: Re10.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_ibb1c2_Coll_bgk_T300_lift.csv
Erfolgreich als CSV gespeichert: Re20.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_drag.csv
Erfolgreich als CSV gespeichert: Re20.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_lift.csv
Erfolgreich als CSV gespeichert: Re20.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_ibb1c2_Coll_bgk_T300_drag.csv
Erfolgreich als CSV gespeichert: Re20.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_ibb1c2_Coll_bgk_T300_lift.csv
Erfolgreich als CSV gespeichert: Re30.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_drag.csv
Erfolgreich als CSV gespeichert: Re30.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_lift.csv
Erfolgreic

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

# 270121 - 163900

In [3]:
# 270121_163900 - Neuste Version

import time
import pandas as pd
import re
import os
import numpy as np
from pathlib import Path

# ==========================================
# 1. HELFER-FUNKTIONEN
# ==========================================

def get_time_range(file_path):
    """Liest Start und Ende der Zeitreihe aus (überspringt Header)."""
    try:
        with open(file_path, 'rb') as f:
            f.readline()  # Header überspringen
            first_line = f.readline().decode().split()
            if not first_line: return None, None
            t_start = float(first_line[1])

            f.seek(0, os.SEEK_END)
            pointer = f.tell() - 2
            while pointer > 0:
                f.seek(pointer)
                if f.read(1) == b'\n': break
                pointer -= 1
            last_line = f.readline().decode().split()
            t_end = float(last_line[1]) if len(last_line) > 1 else t_start
            return t_start, t_end
    except:
        return None, None

def parse_stats_file(file_path):
    """Extrahiert physikalische und numerische Parameter aus der stats.txt."""
    content = file_path.read_text()
    def search(pattern, dtype=float):
        match = re.search(pattern, content, re.IGNORECASE)
        if match:
            val = match.group(1).strip()
            return dtype(val) if dtype else val
        return None

    return {
        'Re': search(r'Re\s*=\s*([\d\.]+)'),
        'Ma': search(r'Ma\s*=\s*([\d\.]+)'),
        'n_steps': search(r'n_steps\s*=\s*(\d+)', int),
        'T_target': search(r'T_target\s*=\s*([\d\.]+)'),
        'GPD': search(r'gridpoints_per_diameter \(gpd\)\s*=\s*(\d+)', int),
        'DpX': search(r'DpX \(D/X\)\s*=\s*([\d\.]+)'),
        'DpY': search(r'DpY \(D/Y\)\s*=\s*([\d\.]+)'),
        'DpZ': search(r'DpZ \(D/Z\)\s*=\s*([\d\.]+)'),
        'shape_LU': search(r'shape_LU:\s*\((.*?)\)', str),
        'gridpoints': search(r'total_number_of_gridpoints:\s*(\d+)', int),
        'bc_type': search(r'bc_type\s*=\s*(\w+)', str),
        'stencil': search(r'stencil\s*=\s*(\w+)', str),
        'collision': search(r'collision\s*=\s*(\w+)', str),
        'tau': search(r'tau\s*=\s*([\d\.]+)'),
        'MLUPS': search(r'MLUPS\s*=\s*([\d\.]+)'),
        'VRAM_peak': search(r'VRAM_peak \[MB\]\s*=\s*([\d\.]+)')
    }

# ==========================================
# 2. HAUPT-FUNKTIONEN (Workflow)
# ==========================================

def build_master_index(base_dir):
    """Scannt Ordner und erstellt den ersten Index."""
    base_path = Path(base_dir)
    records = []
    stats_files = list(base_path.rglob("*_parms_stats_obs.txt"))
    print(f"Verarbeite {len(stats_files)} Simulationen...")

    for sf in stats_files:
        folder = sf.parent
        data = parse_stats_file(sf)
        data['Folder'] = folder.name
        data['path'] = str(folder)
        data['Timestamp'] = sf.name[:13]

        drag_files = list(folder.glob("drag*.txt"))
        if drag_files:
            data['time_start'], data['time_end'] = get_time_range(drag_files[0])
            data['has_drag'] = True
            data['has_lift'] = any(folder.glob("lift*.txt"))
        else:
            data['time_start'] = data['time_end'] = None
            data['has_drag'] = data['has_lift'] = False
        records.append(data)

    return pd.DataFrame(records)

def analyze_simulation_continuity(df):
    """Prüft auf Lücken, Überlappungen und Vollständigkeit."""
    # DpZ entfernt, da es bei 2D-Stencil (D2Q9) oft fehlt (NaN)
    core_params = ['Re', 'Ma', 'GPD', 'DpY', 'bc_type', 'stencil', 'collision']
    df_sorted = df.sort_values(by=core_params + ['time_start']).copy()

    analysis = []
    for name, group in df_sorted.groupby(core_params, dropna=False):
        group = group.reset_index(drop=True)
        for i in range(len(group)):
            row = group.iloc[i].to_dict()
            issues = []
            if i == 0 and row['time_start'] > 0.5:
                issues.append(f"Anfang fehlt (startet bei {row['time_start']})")
            if i > 0:
                gap = row['time_start'] - group.iloc[i-1]['time_end']
                if gap > 0.1: issues.append(f"Lücke: {gap:.2f}s")
                if gap < -0.1: issues.append(f"Überlappung: {abs(gap):.2f}s")
            if i == len(group) - 1:
                if row['T_target'] and row['time_end'] < (row['T_target'] - 1.0):
                    issues.append(f"Abbruch bei {row['time_end']} (Soll: {row['T_target']})")

            row['Status'] = " / ".join(issues) if issues else "OK"
            analysis.append(row)
    return pd.DataFrame(analysis)

def merge_simulation_chains(df_checked, output_base_dir="./merged_data"):
    output_path = Path(output_base_dir)
    output_path.mkdir(exist_ok=True, parents=True)

    core_params = ['Re', 'Ma', 'GPD', 'DpY', 'stencil', 'bc_type', 'collision']
    df_valid = df_checked[df_checked['has_drag'] == True].copy()

    for name, group in df_valid.groupby(core_params, dropna=False):
        group = group.sort_values('time_start')
        first_row, last_row = group.iloc[0], group.iloc[-1]

        t_end_round = int(round(last_row['time_end']))
        name_parts = [
            f"Re{first_row['Re']}", f"Ma{first_row['Ma']}",
            f"GPD{int(first_row['GPD'] or 0)}", f"{first_row['stencil']}",
            f"DpY{first_row['DpY']}", f"BC_{first_row['bc_type']}",
            f"Coll_{first_row['collision']}"
        ]
        if "D3" in str(first_row['stencil']):
            name_parts.append(f"DpZ{first_row['DpZ']}")

        name_parts.append(f"T{t_end_round}")
        base_filename = "_".join(filter(None, [str(p) for p in name_parts]))

        for mode in ['drag', 'lift']:
            combined_chunks = []
            for _, row in group.iterrows():
                folder = Path(row['path'])
                files = list(folder.glob(f"{mode}*.txt"))
                if files:
                    combined_chunks.append(pd.read_csv(files[0], sep=r'\s+'))

            if not combined_chunks: continue

            full_df = pd.concat(combined_chunks, ignore_index=True)

            # Überlapp-Check (5 Nachkommastellen)
            full_df['time_rounded'] = full_df['timePU'].round(5)
            full_df = full_df.drop_duplicates(subset=['time_rounded'], keep='first')
            full_df = full_df.drop(columns=['time_rounded']).sort_values('timePU')

            # Speichern als CSV statt Parquet
            target_file = output_path / f"{base_filename}_{mode}.csv"
            full_df.to_csv(target_file, index=False)
            print(f"Erfolgreich als CSV gespeichert: {target_file.name}")



In [4]:
# ==========================================
# 3. AUSFÜHRUNG
# ==========================================

# Pfade anpassen!
DATA_INPUT = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/00_CP-data_raw"
DATA_OUTPUT = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/01_CP-data_processed"

# Schritt 1: Index erstellen
df_master = build_master_index(DATA_INPUT)

# Schritt 2: Qualität prüfen
df_checked = analyze_simulation_continuity(df_master)
timestamp = time.strftime("%H%M%S")
df_checked.to_csv("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/"+ f"qualitaets_check_{timestamp}.csv", index=False)



Verarbeite 893 Simulationen...


In [ ]:
# Schritt 3: Mergen (nur wenn gewünscht)
merge_simulation_chains(df_checked, DATA_OUTPUT)

In [13]:
# 270121_171800
import pandas as pd
import re
import os
import numpy as np
from pathlib import Path

# ==========================================
# 1. HELFER-FUNKTIONEN
# ==========================================

def get_time_range(file_path):
    """Liest Start und Ende der Zeitreihe aus (überspringt Header)."""
    try:
        with open(file_path, 'rb') as f:
            f.readline()  # Header überspringen
            first_line = f.readline().decode().split()
            if not first_line: return None, None
            # Wir nehmen an, timePU ist das zweite Element (Index 1)
            t_start = float(first_line[1])

            f.seek(0, os.SEEK_END)
            pointer = f.tell() - 2
            while pointer > 0:
                f.seek(pointer)
                if f.read(1) == b'\n': break
                pointer -= 1
            last_line = f.readline().decode().split()
            t_end = float(last_line[1]) if len(last_line) > 1 else t_start
            return t_start, t_end
    except:
        return None, None

def parse_stats_file(file_path):
    """Extrahiert Parameter aus der *_parms_stats_obs.txt."""
    content = file_path.read_text()
    def search(pattern, dtype=float):
        match = re.search(pattern, content, re.IGNORECASE)
        if match:
            val = match.group(1).strip()
            return dtype(val) if dtype else val
        return None

    return {
        'Re': search(r'Re\s*=\s*([\d\.]+)'),
        'Ma': search(r'Ma\s*=\s*([\d\.]+)'),
        'n_steps': search(r'n_steps\s*=\s*(\d+)', int),
        'T_target': search(r'T_target\s*=\s*([\d\.]+)'),
        'GPD': search(r'gridpoints_per_diameter \(gpd\)\s*=\s*(\d+)', int),
        'DpX': search(r'DpX \(D/X\)\s*=\s*([\d\.]+)'),
        'DpY': search(r'DpY \(D/Y\)\s*=\s*([\d\.]+)'),
        'DpZ': search(r'DpZ \(D/Z\)\s*=\s*([\d\.]+)'),
        'bc_type': search(r'bc_type\s*=\s*(\w+)', str),
        'stencil': search(r'stencil\s*=\s*(\w+)', str),
        'collision': search(r'collision\s*=\s*(\w+)', str),
    }

# ==========================================
# 2. HAUPT-FUNKTIONEN
# ==========================================

def build_master_index(base_dir):
    base_path = Path(base_dir)
    records = []
    stats_files = list(base_path.rglob("*_parms_stats_obs.txt"))
    print(f"Scanne {len(stats_files)} Ordner...")

    for sf in stats_files:
        folder = sf.parent
        data = parse_stats_file(sf)
        data['Folder'] = folder.name
        data['path'] = str(folder)

        drag_files = list(folder.glob("drag*.txt"))
        if drag_files:
            data['time_start'], data['time_end'] = get_time_range(drag_files[0])
            data['has_drag'] = True
            data['has_lift'] = any(folder.glob("lift*.txt"))
        else:
            data['time_start'] = data['time_end'] = None
            data['has_drag'] = data['has_lift'] = False
        records.append(data)

    return pd.DataFrame(records)

def analyze_simulation_continuity(df):
    """Gruppiert Simulationen und prüft auf Lücken."""
    core_params = ['Re', 'Ma', 'GPD', 'DpY', 'bc_type', 'stencil', 'collision']
    df_sorted = df.sort_values(by=core_params + ['time_start']).copy()

    analysis = []
    for name, group in df_sorted.groupby(core_params, dropna=False):
        group = group.reset_index(drop=True)
        for i in range(len(group)):
            row = group.iloc[i].to_dict()
            issues = []
            if i == 0 and row['time_start'] and row['time_start'] > 0.5:
                issues.append(f"Start-Lücke ({row['time_start']}s)")
            if i > 0:
                gap = row['time_start'] - group.iloc[i-1]['time_end']
                if gap > 0.1: issues.append(f"Lücke: {gap:.2f}s")
                if gap < -0.1: issues.append(f"Überlapp: {abs(gap):.2f}s")

            row['Status'] = " / ".join(issues) if issues else "OK"
            analysis.append(row)
    return pd.DataFrame(analysis)

def merge_simulation_chains(df_checked, output_base_dir="./merged_results"):
    output_path = Path(output_base_dir)
    output_path.mkdir(exist_ok=True, parents=True)

    # Gruppierungsparameter (ohne DpZ für 2D-Kompatibilität)
    core_params = ['Re', 'Ma', 'GPD', 'DpY', 'stencil', 'bc_type', 'collision']
    df_valid = df_checked[df_checked['has_drag'] == True].copy()

    groups = list(df_valid.groupby(core_params, dropna=False))
    print(f"Starte Merging von {len(groups)} Gruppen...")

    for name, group in groups:
        # Chronologisch sortieren
        group = group.sort_values('time_start')
        first_row, last_row = group.iloc[0], group.iloc[-1]

        # Dateiname bauen
        t_end_round = int(round(last_row['time_end'])) if last_row['time_end'] else 0
        name_parts = [
            f"Re{first_row['Re']}", f"Ma{first_row['Ma']}",
            f"GPD{int(first_row['GPD'] or 0)}", f"{first_row['stencil']}",
            f"DpY{first_row['DpY']}", f"BC_{first_row['bc_type']}",
            f"Coll_{first_row['collision']}"
        ]
        if "D3" in str(first_row['stencil']): name_parts.append(f"DpZ{first_row['DpZ']}")
        name_parts.append(f"T{t_end_round}")
        base_filename = "_".join(filter(None, [str(p) for p in name_parts]))

        for mode in ['drag', 'lift']:
            combined_chunks = []
            target_col = 'Cd' if mode == 'drag' else 'Cl'

            for _, row in group.iterrows():
                folder = Path(row['path'])
                files = list(folder.glob(f"{mode}*.txt"))
                if not files: continue

                try:
                    # DER SCHNELLE PARSER:
                    # Wir ignorieren den Header (Zeilen mit #) komplett.
                    # Wir wissen: Es sind genau 3 Spalten mit Leerzeichen getrennt.
                    df_chunk = pd.read_csv(
                        files[0],
                        sep=r'\s+',
                        engine='c',    # C-Engine für maximale Geschwindigkeit
                        comment='#',   # Ignoriert die Header-Zeile
                        header=None,   # Wir definieren die Namen selbst
                        names=['stepLU', 'timePU', target_col]
                    )

                    if not df_chunk.empty:
                        combined_chunks.append(df_chunk)
                except Exception as e:
                    print(f"Fehler in {files[0].name}: {e}")

            if not combined_chunks: continue

            # Alle Fragmente zusammenkleben
            full_df = pd.concat(combined_chunks, ignore_index=True)

            # Duplikate an den Nahtstellen entfernen (5 Nachkommastellen Toleranz)
            full_df['time_rounded'] = full_df['timePU'].round(5)
            full_df = full_df.drop_duplicates(subset=['time_rounded'], keep='first')
            full_df = full_df.drop(columns=['time_rounded']).sort_values('timePU')

            # Speichern als saubere CSV
            target_file = output_path / f"{base_filename}_{mode}.csv"
            full_df.to_csv(target_file, index=False)
            print(f"Datei fertig: {target_file.name} ({len(full_df)} Zeilen)")

# Ausführung
# merge_simulation_chains(df_checked, "./dein_output_pfad")

In [7]:
# ==========================================
# 3. AUSFÜHRUNG
# ==========================================

# Pfade anpassen!
DATA_INPUT = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/00_CP-data_raw"
DATA_OUTPUT = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/01_CP-data_processed"

# Schritt 1: Index erstellen
df_master = build_master_index(DATA_INPUT)

# Schritt 2: Qualität prüfen
df_checked = analyze_simulation_continuity(df_master)
timestamp = time.strftime("%H%M%S")
df_checked.to_csv("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/"+ f"qualitaets_check_{timestamp}.csv", index=False)


Scanne 893 Ordner...


In [14]:

# 3. Merge
merge_simulation_chains(df_checked, DATA_OUTPUT)

Starte Merging von 848 Gruppen...
Datei fertig: Re10.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_drag.csv (519616 Zeilen)
Datei fertig: Re10.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_lift.csv (519616 Zeilen)
Datei fertig: Re10.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_ibb1c2_Coll_bgk_T300_drag.csv (519616 Zeilen)
Datei fertig: Re10.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_ibb1c2_Coll_bgk_T300_lift.csv (519616 Zeilen)
Datei fertig: Re20.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_drag.csv (519616 Zeilen)
Datei fertig: Re20.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_lift.csv (519616 Zeilen)
Datei fertig: Re20.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_ibb1c2_Coll_bgk_T300_drag.csv (519616 Zeilen)
Datei fertig: Re20.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_ibb1c2_Coll_bgk_T300_lift.csv (519616 Zeilen)
Datei fertig: Re30.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_drag.csv (519616 Zeilen)
Datei fertig: Re30.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_lift.csv (519616 Zeilen)
Datei 

#### Check auf Unstetigkeiten


In [15]:
# 270121 - 174900
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def scan_all_merges_for_jumps(merged_dir, rel_threshold=0.05, abs_threshold=0.005):
    """
    Scannt alle CSVs im Ordner auf Sprünge an den Nahtstellen.
    rel_threshold: 0.05 entspricht 5% Abweichung zum Vorwert.
    abs_threshold: Mindest-Absolutwert (um Rauschen bei Werten nahe 0 zu ignorieren).
    """
    merged_path = Path(merged_dir)
    csv_files = list(merged_path.glob("*.csv"))
    report = []

    print(f"Scanne {len(csv_files)} Dateien auf Unstetigkeiten...")

    for file in csv_files:
        df = pd.read_csv(file)
        val_col = 'Cd' if 'drag' in file.name else 'Cl'

        # Nahtstellen finden (wo stepLU sinkt oder gleich bleibt)
        transition_indices = df.index[df['stepLU'].diff() <= 0].tolist()

        for idx in transition_indices:
            # Wir brauchen den Wert direkt davor (idx-1) und den an der Naht (idx)
            val_before = df.loc[idx-1, val_col]
            val_after = df.loc[idx, val_col]

            diff = abs(val_after - val_before)

            # Relative Abweichung berechnen (Vorsicht bei Division durch ~0)
            denom = abs(val_before) if abs(val_before) > 1e-6 else 1e-6
            rel_diff = diff / denom

            # Nur melden, wenn sowohl relativer als auch absoluter Sprung relevant sind
            if rel_diff > rel_threshold and diff > abs_threshold:
                error_data = {
                    'file': file.name,
                    'time': df.loc[idx, 'timePU'],
                    'val_before': val_before,
                    'val_after': val_after,
                    'jump_pct': rel_diff * 100
                }
                report.append(error_data)
                print(f"⚠️ Sprung in {file.name}: {rel_diff:.1%} bei t={df.loc[idx, 'timePU']}")

    return pd.DataFrame(report)

# --- AUTOMATISCHE PLOT-FUNKTION FÜR PROBLEMFAELLE ---
def plot_problem_cases(report_df, merged_dir, window=100):
    """Erstellt Plots nur für die Dateien, in denen Sprünge gefunden wurden."""
    for _, row in report_df.iterrows():
        file_path = Path(merged_dir) / row['file']
        df = pd.read_csv(file_path)
        val_col = 'Cd' if 'drag' in row['file'] else 'Cl'

        # Finde den Index für die Zeit im Report
        idx = df.index[df['timePU'] == row['time']][0]

        start, end = max(0, idx-window), min(len(df), idx+window)
        subset = df.iloc[start:end]

        plt.figure(figsize=(8, 4))
        plt.plot(subset['timePU'], subset[val_col], 'b-o', markersize=2, label='Daten')
        plt.axvline(row['time'], color='red', linestyle='--', label='Nahtstelle')
        plt.title(f"Sprung-Detail: {row['file']}\nZeit: {row['time']:.2f}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

# Ausführung:
# df_jumps = scan_all_merges_for_jumps("./merged_results")
# if not df_jumps.empty:
#     plot_problem_cases(df_jumps, "./merged_results")

In [17]:
df_jumps = scan_all_merges_for_jumps(DATA_OUTPUT)

Scanne 1696 Dateien auf Unstetigkeiten...


#### finale Master-Übersicht erstellen (DATAFRAME:

In [18]:
# 260121_180400 - Master-DF, der alle relevantenn Datenreihen benennt
def create_final_summary(df_checked):
    # Definition der Spalten, die in die Übersicht sollen
    summary_cols = [
        'Re', 'Ma', 'n_steps', 'T_target', 'GPD',
        'DpX', 'DpY', 'DpZ', 'bc_type', 'stencil', 'collision'
    ]

    # Wir gruppieren nach diesen Parametern.
    # .first() nimmt die Werte der ersten Simulation in der Kette (die sind identisch)
    # .max() bei time_end liefert uns die tatsächlich erreichte Zeit der gesamten Kette
    summary_df = df_checked.groupby(summary_cols, dropna=False).agg({
        'time_end': 'max',
        'path': 'count' # Hilfreich um zu sehen, aus wie vielen Fragmenten die Kette bestand
    }).reset_index()

    # Spalten umbenennen für mehr Klarheit
    summary_df = summary_df.rename(columns={'path': 'num_fragments'})

    # Sortieren für bessere Lesbarkeit (z.B. nach Reynolds-Zahl)
    summary_df = summary_df.sort_values(['Re', 'Ma', 'GPD'])

    return summary_df

In [20]:
# --- AUSFÜHRUNG ---
final_summary = create_final_summary(df_checked)

# Speichern als CSV für Dokumentation oder Excel
final_summary.to_csv("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/"+ "simulation_master_overview.csv", index=False)

# Anzeige der ersten Zeilen
print("Master-Übersicht erstellt:")
print(final_summary.head())

Master-Übersicht erstellt:
     Re    Ma  n_steps   T_target  GPD    DpX   DpY  DpZ bc_type stencil  \
0  10.0  0.05   519615  299.99986   50  100.0  50.0  NaN  hwbbc2    D2Q9   
1  10.0  0.05   519615  299.99986   50  100.0  50.0  NaN  ibb1c2    D2Q9   
2  20.0  0.05   519615  299.99986   50  100.0  50.0  NaN  hwbbc2    D2Q9   
3  20.0  0.05   519615  299.99986   50  100.0  50.0  NaN  ibb1c2    D2Q9   
4  30.0  0.05   519615  299.99986   50  100.0  50.0  NaN  hwbbc2    D2Q9   

  collision   time_end  num_fragments  
0       bgk  299.99986              1  
1       bgk  299.99986              1  
2       bgk  299.99986              1  
3       bgk  299.99986              1  
4       bgk  299.99986              1  


In [21]:
# 260121_1817 - Zwischenspeicher
import pandas as pd
from pathlib import Path

# Dateipfade für die Checkpoints
FILE_RAW_INDEX = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/master_index_raw.csv"
FILE_CHECKED_INDEX = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/master_index_checked.csv"

def get_or_create_index(data_input_dir, force_rescan=False):
    # 1. Prüfen, ob der analysierte Index schon da ist
    if Path(FILE_CHECKED_INDEX).exists() and not force_rescan:
        print(f"Lade existierenden Analyse-Index aus {FILE_CHECKED_INDEX}...")
        return pd.read_csv(FILE_CHECKED_INDEX)

    # 2. Falls nicht, prüfen ob zumindest der rohe Scan da ist
    if Path(FILE_RAW_INDEX).exists() and not force_rescan:
        print(f"Lade rohen Scan aus {FILE_RAW_INDEX} und starte Analyse...")
        df_raw = pd.read_csv(FILE_RAW_INDEX)
    else:
        # 3. Komplett neu scannen (dauert am längsten)
        print("Scanne Verzeichnisse neu (das kann dauern)...")
        df_raw = build_master_index(data_input_dir)
        df_raw.to_csv(FILE_RAW_INDEX, index=False)

    # Analyse durchführen
    print("Führe Kontinuitätsanalyse durch...")
    df_checked = analyze_simulation_continuity(df_raw)
    df_checked.to_csv(FILE_CHECKED_INDEX, index=False)

    return df_checked

In [ ]:
# --- ANWENDUNG IM HAUPTPROGRAMM ---

# Pfade
INPUT_DIR = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/00_CP-data_raw"
OUTPUT_MERGED = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/01_CP-data_processed"

# Holt den Index (entweder von Festplatte oder durch neuen Scan)
df_checked = get_or_create_index(INPUT_DIR, force_rescan=False)

# Jetzt kannst du direkt damit arbeiten:
# final_summary = create_final_summary(df_checked)

# Das Mergen musst du nur einmal ausführen. Danach liegen die fertigen
# Zeitreihen ja bereits in OUTPUT_MERGED.
# merge_simulation_chains(df_checked, OUTPUT_MERGED)

In [ ]:
# 260121_182800
import pandas as pd
import re
from pathlib import Path

def create_summary_from_merged_files(merged_dir):
    merged_path = Path(merged_dir)
    # Wir suchen nur nach den drag-Dateien (da lift meist identisch ist)
    files = list(merged_path.glob("*_drag.csv"))

    summary_data = []

    print(f"Analysiere {len(files)} Master-Dateien...")

    for f in files:
        # 1. Metadaten aus dem Dateinamen extrahieren
        # Beispiel: Re100.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_drag.csv
        name = f.stem  # Dateiname ohne .csv

        def extract(pattern, string):
            match = re.search(pattern, string)
            return match.group(1) if match else None

        # 2. Die Datei kurz einlesen, um echte Werte zu kriegen
        # Wir lesen nur die erste und letzte Zeile für Speed
        try:
            # Wir nutzen dask oder nur skiprows/nrows um nicht
            # die ganze Riesen-Datei zu laden
            first_row = pd.read_csv(f, nrows=1)
            # Für die letzte Zeile:
            df_full = pd.read_csv(f) # Falls Dateien nicht Giga-Byte groß sind

            last_time = df_full['timePU'].max()
            row_count = len(df_full)

            summary_data.append({
                'Re': float(extract(r'Re([\d\.]+)', name) or 0),
                'Ma': float(extract(r'Ma([\d\.]+)', name) or 0),
                'GPD': int(extract(r'GPD(\d+)', name) or 0),
                'stencil': extract(r'_(D\dQ\d+)_', name),
                'DpY': float(extract(r'DpY([\d\.]+)', name) or 0),
                'DpZ': float(extract(r'DpZ([\d\.]+)', name) or 0) if "D3" in name else None,
                'bc_type': extract(r'BC_([^_]+)', name),
                'collision': extract(r'Coll_([^_]+)', name),
                'real_time_end': last_time,
                'total_steps_recorded': row_count,
                'file_name': f.name
            })
        except Exception as e:
            print(f"Fehler bei Datei {f.name}: {e}")

    df_final = pd.DataFrame(summary_data)

    # Sortieren für die Übersicht
    df_final = df_final.sort_values(['Re', 'Ma', 'GPD'])

    return df_final

# --- ANWENDUNG ---
# final_master_summary = create_summary_from_merged_files("./merged_data")
# final_master_summary.to_csv("final_production_overview.csv", index=False)

In [1]:
# 260211_160000: Erster Versuch einfacher Analysefunktion, wird aber nicht verwendet:

def analyze_aerodynamics(df, t_start_stat, plot=True):
    """
    Berechnet Mittelwerte und Standardabweichung ab einem Zeitpunkt t_start_stat.
    Erstellt optional einen Plot mit dem gemittelten Bereich.
    """
    # 1. Daten filtern (Einschwingvorgang abschneiden)
    df_stat = df[df['timePU'] >= t_start_stat]

    if df_stat.empty:
        print(f"Warnung: Keine Daten nach t={t_start_stat} vorhanden.")
        return None

    # 2. Statistik berechnen
    stats = {}
    for col in ['Cd', 'Cl']:
        if col in df.columns:
            stats[f'mean_{col}'] = df_stat[col].mean()
            stats[f'std_{col}'] = df_stat[col].std()
            stats[f'max_{col}'] = df_stat[col].max()
            stats[f'min_{col}'] = df_stat[col].min()

    # 3. Plotten (Visuelle Kontrolle)
    if plot:
        # Wir nehmen Cd oder Cl, je nachdem was da ist
        val_col = 'Cd' if 'Cd' in df.columns else 'Cl'

        plt.figure(figsize=(10, 5))

        # Grauer Bereich: Ignorierte Daten (Einschwingen)
        df_ignored = df[df['timePU'] < t_start_stat]
        plt.plot(df_ignored['timePU'], df_ignored[val_col], color='gray', alpha=0.5, label='Ignoriert (Transient)')

        # Blauer Bereich: Statistik-Daten
        plt.plot(df_stat['timePU'], df_stat[val_col], color='blue', label='Analyse-Bereich')

        # Mittelwert-Linie
        mean_val = stats[f'mean_{val_col}']
        plt.axhline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.4f}')

        plt.title(f"Analyse {val_col} (Statistik ab t={t_start_stat})")
        plt.xlabel("Time [PU]")
        plt.ylabel(val_col)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    return stats

In [1]:
# 260225 - PLOTTING STATS

def plot_paper_figure(stats_df, series_list, global_settings=None):
    """
    Erstellt hochgradig anpassbare Plots für Publikationen.

    :param stats_df: Dein DataFrame mit den Statistik-Ergebnissen.
    :param series_list: Eine Liste von Dictionaries. Jedes Dict definiert eine Kurve.
    :param global_settings: Dictionary für globale Achsen-Einstellungen.
    """
    plt.figure(figsize=(10, 6))

    # 1. Jede Datenreihe einzeln verarbeiten und plotten
    for i, series in enumerate(series_list):
        df_sub = stats_df.copy()
        criteria = series.get('criteria', {})

        # Daten filtern
        for col, val in criteria.items():
            if isinstance(val, (list, tuple)):
                df_sub = df_sub[df_sub[col].isin(val)]
            else:
                df_sub = df_sub[df_sub[col] == val]

        if df_sub.empty:
            print(f"⚠️ Warnung: Keine Daten gefunden für Reihe {i+1} mit Kriterien: {criteria}")
            continue

        # Achsen-Spalten holen
        x_col = series.get('x')
        y_col = series.get('y')
        yerr_col = series.get('yerr', None)

        # WICHTIG: Nach X-Werten sortieren, damit die Linien nicht im Zickzack verlaufen
        df_sub = df_sub.sort_values(by=x_col)

        # Styling-Parameter (Farbe, Marker, etc.)
        style = series.get('style', {})

        # Default capsize für Fehlerbalken, falls nicht überschrieben
        if 'capsize' not in style and yerr_col:
            style['capsize'] = 4

        # Plotten (mit oder ohne Fehlerbalken)
        if yerr_col and yerr_col in df_sub.columns:
            # errorbar nutzt die Style-Parameter
            plt.errorbar(df_sub[x_col], df_sub[y_col], yerr=df_sub[yerr_col], **style)
        else:
            # normales plot nutzt ebenfalls die Style-Parameter
            plt.plot(df_sub[x_col], df_sub[y_col], **style)

    # 2. Globale Plot-Einstellungen anwenden (Titel, Achsen, Skalierung)
    if global_settings is None:
        global_settings = {}

    if 'title' in global_settings:
        plt.title(global_settings['title'], fontsize=14)
    if 'xlabel' in global_settings:
        plt.xlabel(global_settings['xlabel'], fontsize=12)
    if 'ylabel' in global_settings:
        plt.ylabel(global_settings['ylabel'], fontsize=12)

    if global_settings.get('xscale') == 'log':
        plt.xscale('log')
    if global_settings.get('yscale') == 'log':
        plt.yscale('log')

    if 'xlim' in global_settings:
        plt.xlim(global_settings['xlim'])
    if 'ylim' in global_settings:
        plt.ylim(global_settings['ylim'])

    # Grid und Legende
    plt.grid(True, alpha=0.3, which='both', linestyle='--')

    # Legenden-Position anpassbar machen
    loc = global_settings.get('legend_loc', 'best')
    plt.legend(loc=loc, fontsize=11)

    plt.tight_layout()
    plt.show()

In [ ]:
# 1. Wir definieren exakt, welche Kurven in den Plot sollen
meine_kurven = [
    {
        'criteria': {'stencil': 'D2Q9', 'Re': 40, 'DpY': 19, 'collision': 'bgk', 'bc_type': ['fwbb', 'fwbbc']},
        'x': 'GPD',
        'y': 'Cd_mean',
        'yerr': 'Cd_stdev', # Die Fehlerbalken
        'style': {
            'label': 'FWBB BGK',
            'color': 'tab:blue',
            'marker': '.',
            'linestyle': '--',
            #'linewidth': 1.5,
            #'markersize': 6
            }
    },
    {
        'criteria': {'stencil': 'D2Q9', 'Re': 40, 'DpY': 19, 'collision': 'kbc', 'bc_type': ['fwbb', 'fwbbc']},
        'x': 'GPD',
        'y': 'Cd_mean',
        'yerr': 'Cd_stdev', # Die Fehlerbalken
        'style': {
            'label': 'FWBB KBC',
            'color': 'tab:blue',
            'marker': 'x',
            'linestyle': '--',
            #'linewidth': 1.5,
            #'markersize': 6
            }
    },
    {
        'criteria': {'stencil': 'D2Q9', 'Re': 40, 'DpY': 19, 'collision': 'kbc', 'bc_type': ['hwbb', 'hwbbc2']},
        'x': 'GPD',
        'y': 'Cd_mean',
        'yerr': 'Cd_stdev', # Die Fehlerbalken
        'style': {
            'label': 'HWBB KBC',
            'color': 'tab:orange',
            'marker': 'x',
            'linestyle': '--',
            #'linewidth': 1.5,
            #'markersize': 6
            }
    },
    {
        'criteria': {'stencil': 'D2Q9', 'Re': 40, 'DpY': 19, 'collision': 'bgk', 'bc_type': ['ibb1', 'ibb1c2']},
        'x': 'GPD',
        'y': 'Cd_mean',
        'yerr': 'Cd_stdev', # Die Fehlerbalken
        'style': {
            'label': 'IBB BGK',
            'color': 'tab:green',
            'marker': '.',
            'linestyle': '--',
            #'linewidth': 1.5,
            #'markersize': 6
            }
    },
    {
        'criteria': {'stencil': 'D2Q9', 'Re': 40, 'DpY': 19, 'collision': 'kbc', 'bc_type': ['ibb1', 'ibb1c2']},
        'x': 'GPD',
        'y': 'Cd_mean',
        'yerr': 'Cd_stdev', # Die Fehlerbalken
        'style': {
            'label': 'IBB KBC',
            'color': 'tab:green',
            'marker': 'x',
            'linestyle': '--',
            #'linewidth': 1.5,
            #'markersize': 6
            }
    }
]

# 2. Wie soll der Rahmen des Plots aussehen?
globale_einstellungen = {
    #'title': 'Mean Drag Coefficient over GPD',
    'xlabel': 'GPD',
    'ylabel': r'$C_{D,mean}$', # r'' erlaubt LaTeX-Formatierung
    #'xscale': 'log',                          # Für Re macht Log oft Sinn
    #'legend_loc': 'upper right',
    'xlim': (9,61),
    'linestyle': '--',
    'font.size': 7,
    'linewidth': 1,
    'markeredgewidth': 1,
    'markersize': 5,
    'figsize': (3.4876,2.301816),
    'autolayout': True,
    'constrained_layout': True,
    'dpi': 300
}

# 3. Den Plot generieren!
plot_paper_figure(df_stats, meine_kurven, globale_einstellungen)